# DATA

## EXTRACCION DE DATOS
Para el desarrollo de este proyecto se recurrió a dos tipos de fuentes. Por un lado, las pistas de audio utilizadas para construir el dataset fueron obtenidas de Free Music Archive (https://freemusicarchive.org), plataforma de distribución musical que ofrece canciones bajo licencias Creative Commons, permitiendo su uso libre en contextos académicos e investigativos sin restricciones de derechos de autor. Las canciones fueron seleccionadas y organizadas manualmente por género musical, garantizando un mínimo de 50 pistas por categoría.

## PROCESAMIENTO Y MODELAMIENTO DE LOS DATOS

### Librerias 
Librosa: es una librería especializada en análisis de audio y música que permite cargar archivos de sonido y extraer características acústicas. Es el núcleo del proceso de extracción de features en este proyecto.

NumPy: es la librería fundamental para computación numérica en Python. Proporciona estructuras de datos tipo array de alto rendimiento y operaciones matemáticas vectorizadas, siendo la base sobre la que trabajan la mayoría de librerías científicas incluyendo Librosa.

Pandas es la librería estándar para manipulación y análisis de datos tabulares en Python. Permite construir, limpiar y transformar dataframes, que es la estructura utilizada para almacenar y exportar el dataset final en formato CSV.

Os es un módulo nativo de Python que permite interactuar con el sistema operativo, específicamente con el sistema de archivos. En este proyecto se utiliza para recorrer las carpetas de géneros, listar los archivos de audio disponibles y construir las rutas de acceso a cada canción.

In [1]:
import librosa
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

### Extracción de Features 
El código implementa un pipeline de extracción automática de características acústicas a partir de archivos de audio organizados por género musical. Para cada canción se carga la pista completa y se extraen tres segmentos de 30 segundos correspondientes al inicio, la mitad y el final de la pista. De cada segmento se calculan 27 features: siete características acústicas globales (tempo, spectral centroid, spectral bandwidth, rolloff, zero crossing rate, RMS energy y chroma STFT) y 20 coeficientes MFCC que representan el timbre de la señal. Cada segmento procesado queda registrado como una fila independiente en el dataset, junto con su etiqueta de género y el nombre del archivo de origen. El resultado final es un archivo CSV estructurado y listo para la etapa de modelado.

In [ ]:
def extraerfeatures(y, sr):
    mfccs      = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=5) # solo los 5 primeros coeficientes
    onset_env  = librosa.onset.onset_strength(y=y, sr=sr) # fuerza de los inicios (ritmo)
    chroma     = librosa.feature.chroma_stft(y=y, sr=sr) # 12 notas × frames
    features = {
        'tempo':               librosa.beat.tempo(y=y, sr=sr)[0],  # BPM
        'spectral_centroid':   librosa.feature.spectral_centroid(y=y, sr=sr).mean(),
        'spectral_bandwidth':  librosa.feature.spectral_bandwidth(y=y, sr=sr).mean(),
        'rolloff':             librosa.feature.spectral_rolloff(y=y, sr=sr).mean(),
        'zero_crossing_rate':  librosa.feature.zero_crossing_rate(y).mean(),
        'rms':                 librosa.feature.rms(y=y).mean(),
        'chroma_stft':         chroma.mean(),
        'onset_strength_mean': onset_env.mean(), # ritmo promedio
        'onset_strength_std':  onset_env.std(), # regularidad del ritmo
        'pitch_variance':      chroma.var(), # estabilidad armónica
    }
    for i, coef in enumerate(mfccs):
        features[f'mfcc{i+1}_mean'] = coef.mean() # timbre promedio
        features[f'mfcc{i+1}_std']  = coef.std() # variabilidad del timbre
    return features

def obtenersegmentos(duracion_total):
    inicio = 0
    mitad  = int(duracion_total / 2) - 15
    final  = int(duracion_total) - 30
    return {'inicio': inicio, 'mitad': mitad, 'final': final}

filas = []
carpeta_raiz = r'C:\Machine Learning project 1\Musica' # Ruta de la carpeta raiz que contiene las subcarpetas de géneros musicales

# Diagnóstico inicial
print(f"Carpeta raíz existe: {os.path.exists(carpeta_raiz)}")
print(f"Contenido de la carpeta raíz: {os.listdir(carpeta_raiz)}")

for genero in os.listdir(carpeta_raiz):
    carpeta_genero = os.path.join(carpeta_raiz, genero)
    if not os.path.isdir(carpeta_genero):
        print(f"  SKIP (no es carpeta): {genero}")
        continue

    archivos_genero = os.listdir(carpeta_genero)
    print(f"\n{genero}: {len(archivos_genero)} archivos → {archivos_genero[:3]}")  # muestra los primeros 3

    for archivo in archivos_genero:
        if not (archivo.endswith('.mp3') or archivo.endswith('.wav')):
            print(f"  SKIP (formato no válido): {archivo}")
            continue

        ruta = os.path.join(carpeta_genero, archivo)
        try:
            cancion, sr = librosa.load(ruta, sr=None)
            duracion_total = librosa.get_duration(y=cancion, sr=sr)

            if duracion_total < 90:
                print(f"  SKIP (muy corta {duracion_total:.0f}s): {archivo}")
                continue

            segmentos = obtenersegmentos(duracion_total)

            for nombre_seg, seg_inicio in segmentos.items():
                inicio_muestra = int(seg_inicio * sr)
                fin_muestra    = int((seg_inicio + 30) * sr)
                segmento       = cancion[inicio_muestra:fin_muestra]

                features = extraerfeatures(segmento, sr)
                features['label']    = genero
                features['cancion']  = archivo
                features['segmento'] = nombre_seg
                filas.append(features)

            print(f"  OK: {archivo} ({duracion_total:.0f}s)")

        except Exception as e:
            print(f"  ERROR en {archivo}: {e}")

# Verificación antes de guardar
if len(filas) == 0:
    print("\nERROR: no se procesó ningún archivo. Revisa los SKIP y ERROR arriba.")
else:
    df = pd.DataFrame(filas)
    df.to_csv('dataset_musical.csv', index=False)
    print(f"\nDataset guardado: {df.shape}")
    print(f"Muestras por género:\n{df['label'].value_counts()}")

Carpeta raíz existe: True
Contenido de la carpeta raíz: ['Rock']

Rock: 50 archivos → ['1st Contact - Bloody Sun.mp3', 'Abunai! - Barbara Allen.mp3', 'Abunai! - Inspiration.mp3']
  OK: 1st Contact - Bloody Sun.mp3 (386s)
  OK: Abunai! - Barbara Allen.mp3 (298s)
  OK: Abunai! - Inspiration.mp3 (320s)
  OK: Adeline Yeo (HP) - Innovative Ocean Music Melody Combination - dwilly And Adeline Yeo (HP).mp3 (218s)
  OK: Alex-Productions - Extreme Rock aggressive _ Punch.mp3 (100s)
  OK: eddy - Machinery.mp3 (221s)
  SKIP (muy corta 46s): eddy - Ruff.mp3
  OK: Here Comes A Big Black Cloud!! - The Fly Pt. II.mp3 (144s)
  OK: Jasmine Love Bomb - Empire Sun.mp3 (259s)
  OK: Jasmine Love Bomb - That River.mp3 (358s)
  OK: Jon Shuemaker - Don't Let Your Mind Play Tricks On You.mp3 (225s)
  OK: Jon Shuemaker - Don't Throw Me Overboard.mp3 (279s)
  OK: Jon Shuemaker - Early Grave.mp3 (242s)
  OK: Jon Shuemaker - Face the Day.mp3 (142s)
  OK: Jon Shuemaker - Fire & Water.mp3 (260s)
  OK: Jon Shuemaker -

## RESULTADOS

In [5]:
x = pd.read_csv('dataset_musical.csv')
x

,tempo,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,rms,chroma_stft,onset_strength_mean,onset_strength_std,pitch_variance,...,mfcc2_std,mfcc3_mean,mfcc3_std,mfcc4_mean,mfcc4_std,mfcc5_mean,mfcc5_std,label,cancion,segmento
0,129.199219,3545.709923,4484.520723,8468.857123,0.054472,0.151978,0.586850,1.071236,1.350803,0.058771,...,29.592175,17.636984,30.214000,33.475770,15.048951,5.258302,17.074522,Rock,1st Contact - Bloody Sun.mp3,inicio
1,129.199219,3615.850166,4653.082267,8610.514600,0.055159,0.173410,0.572499,1.145310,1.056980,0.064809,...,28.579472,7.005992,17.644274,26.491468,9.662541,2.247495,9.909929,Rock,1st Contact - Bloody Sun.mp3,mitad
2,129.199219,4978.427223,5568.908509,12210.167833,0.101688,0.161031,0.579035,0.964773,0.663614,0.060304,...,28.670221,6.593816,18.909647,21.428247,12.932837,7.282473,9.651064,Rock,1st Contact - Bloody Sun.mp3,final
3,126.048018,5972.016916,3599.401077,9795.590767,0.278675,0.093559,0.387640,0.844744,0.193384,0.080643,...,46.691200,-59.654090,31.732536,52.668453,22.213177,-48.078003,28.067030,Rock,Abunai! - Barbara Allen.mp3,inicio
4,139.674831,2808.053122,3246.018162,5930.505806,0.064733,0.189166,0.488053,1.030586,0.465280,0.076586,...,18.774284,-41.099560,12.523503,44.248344,12.540815,-18.917416,10.016224,Rock,Abunai! - Barbara Allen.mp3,mitad
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136,132.512019,2634.998983,3090.828091,4835.220762,0.065590,0.147627,0.488243,1.066565,0.597017,0.073513,...,26.177948,-51.838670,12.675660,17.855246,12.611792,-5.803196,11.474655,Rock,The Men - Open Your Heart.mp3,mitad
137,126.048018,3255.230293,3496.967991,6203.820820,0.079176,0.104300,0.546732,0.948762,0.347108,0.067545,...,25.180845,-46.032513,15.361878,41.128483,15.420674,-1.295216,13.714457,Rock,The Men - Open Your Heart.mp3,final
138,120.185320,1330.955576,1976.759476,2544.826278,0.024744,0.211475,0.500139,0.996439,0.934447,0.084275,...,35.075394,-10.500660,18.432049,17.476114,31.522621,-4.474307,14.813031,Rock,The New Lines - Burning Bridges.mp3,inicio
139,123.046875,1504.444413,2196.362643,3067.739783,0.024896,0.226691,0.484661,1.004126,1.002821,0.074712,...,26.480997,-28.441395,16.217490,36.556763,14.282331,2.036050,10.518405,Rock,The New Lines - Burning Bridges.mp3,mitad


Tempo
Mide los BPM (pulsaciones por minuto) de la canción. Librosa detecta los picos de energía que se repiten en el tiempo y calcula cada cuánto ocurren. Un género como el metal o el disco tiene tempo alto, el blues o el classical lo tiene bajo.

Spectral Centroid
Es el centro de gravedad del espectro de frecuencias. Se calcula como el promedio ponderado de todas las frecuencias presentes en la señal. Un valor alto significa que la energía está concentrada en frecuencias agudas (sonido brillante), uno bajo en frecuencias graves (sonido oscuro y profundo).

Spectral Bandwidth
Mide qué tan dispersas están las frecuencias alrededor del centroid. Si el ancho es grande, la canción tiene mucho contenido en un rango amplio de frecuencias. Si es pequeño, el sonido es más puro y concentrado. Géneros ruidosos como el metal tienen bandwidth alto.

Rolloff
Es la frecuencia por debajo de la cual se encuentra el 85% de la energía total de la señal. Sirve para detectar si una canción tiene mucho contenido en frecuencias altas o bajas. Distingue bien géneros con mucho agudo (rock, metal) de géneros más cálidos (jazz, blues).

Zero Crossing Rate
Cuenta cuántas veces por segundo la señal de audio pasa de un valor positivo a uno negativo. Una señal percusiva o ruidosa cruza el cero muy frecuentemente. Una señal tonal y melódica lo hace pocas veces. Es útil para separar géneros como hiphop y metal de classical y jazz.

RMS (Root Mean Square Energy)
Es la raíz cuadrada del promedio de los cuadrados de las amplitudes de la señal. Representa el volumen real promedio de la canción. Géneros intensos como metal tienen RMS alto, géneros acústicos o suaves como classical lo tienen bajo.

Chroma STFT
Mapea todas las frecuencias de la señal a las 12 notas musicales de la escala cromática (do, do#, re, re#...) y mide cuánto aparece cada una. Captura la armonía y la tonalidad de la canción. Es especialmente útil para distinguir géneros que tienen estructuras armónicas características, como el jazz o el reggae.

onset_strength_mean
Es el promedio de la fuerza con la que ocurren los golpes rítmicos a lo largo del segmento. Un valor alto indica que los beats son intensos y pronunciados, como en el hiphop o el metal. Un valor bajo indica golpes suaves o difusos, como en el classical.

onset_strength_std
Es la desviación estándar de esa fuerza. Mide qué tan regular o irregular es el ritmo. Un valor bajo significa que los beats ocurren con intensidad constante (ritmo mecánico y predecible como el disco o el hiphop). Un valor alto significa que la intensidad varía mucho entre beats (ritmo irregular y expresivo como el jazz).

pitch_variance
Es la varianza del contenido armónico de la canción calculada sobre el histograma de chroma. Mide qué tan estable es la armonía. Un valor bajo indica que la canción se mantiene en pocas notas o tonalidades (reggae, pop). Un valor alto indica que la canción transita por muchas notas distintas con frecuencia (jazz, classical).

mfccX_mean
El promedio del coeficiente MFCC número X a lo largo del segmento. Describe el timbre promedio de la canción en esa dimensión espectral. El coeficiente 1 captura la energía general, los siguientes capturan detalles progresivamente más finos del timbre.

mfccX_std
La desviación estándar del mismo coeficiente. Mide qué tan variable fue ese aspecto del timbre durante el segmento. Un valor alto significa que el sonido cambia mucho en esa dimensión, uno bajo que se mantiene estable. Esta columna es nueva respecto al código original y aporta información sobre la dinámica interna del segmento que la media sola no captura.

label
Indica el genero de la cancion

cancion
Indica el nombre del archivo

Segmento
Indica cual de los 3 segmentos representa la fila